![Databricks Academy](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/db-academy.png)

# 12 - Change Data Capture with AUTO CDC with Slowly Changing Dimensions (SCD) TYPE 1

In this demonstration, we will continue to build our pipeline by ingesting **customer** data. The customer data includes new customers, customers who have deleted their accounts, and customers who have updated their information (such as address, email, etc.). We will need to build our customer pipeline by implementing change data capture (CDC) for customer data using SCD Type 1 (Type 2 is outside the scope of this course).

The customer pipeline flow:

- The bronze table uses **Auto Loader** to ingest JSON data from cloud object storage with SQL (`FROM STREAM`).
- A table is defined to enforce constraints before passing records to the silver layer.
- `AUTO CDC` is used to automatically process CDC data into the silver layer as a Type 1.
- A gold table is defined to create a materialized view of the current customers with updated information (dropped customers, new customers and updated customer information).



### Learning Objectives

By the end of this lesson, you will be able to:
- Apply the `AUTO CDC` operation in Apache Spark™ Declarative Pipelines to process change data capture (CDC) by integrating and updating incoming data from a source stream into an existing UC table, ensuring data accuracy and consistency.
- Analyze Slowly Changing Dimensions (SCD Type 1) tables within Apache Spark™ Declarative Pipelines to effectively update, insert, and drop customers in dimensional data, managing the state of records over time using appropriate keys, versioning, and timestamps.


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Syntax Update</strong>
  <div style="color:#333;">

The AUTO CDC APIs replace the APPLY CHANGES APIs, and have the same syntax. The APPLY CHANGES APIs are still available, but Databricks recommends using the AUTO CDC APIs in their place.

The AUTO CDC APIs - Simplify change data capture with pipelines:
[AWS](https://docs.databricks.com/aws/en/ldp/cdc) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc) |
[GCP](https://docs.databricks.com/gcp/en/ldp/cdc)


  </div>
</div>



## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the following cell to configure your working environment for this course.

    This cell will also reset your `/Volumes/labuser/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

## B. Explore the Customer Data Source Files

1. Run the cell below to programmatically view the files in your `/Volumes/labuser/sdp_1_bronze/source/customers` volume. 

    Confirm you only see one **00.json** file for customers.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/customers"').display()

2. Run the query below to explore the customers **00.json** file located at `/Volumes/labuser/sdp_1_bronze/source/customers`. Note the following:

   a. The file contains **939 customers** (remember this number).

   b. It includes general customer information such as **email**, **name**, and **address**.

   c. The **timestamp** column specifies the logical order of customer events in the source data.

   d. The **operation** column indicates whether the entry is for a new customer, a deletion, or an update.
      - **NOTE:** Since this is the first JSON file, all rows will be considered new customers.


In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/customers/00.json',
  format => "JSON"
)
ORDER BY operation;


<div style="
  border-left: 4px solid #7b1fa2;
  background: #f3e5f5;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#4a148c; margin-bottom:6px; font-size: 1.1em;">Question</strong>
  <div style="color:#333;">

How can we ingest new raw data source files (JSON) with customer updates into our pipeline to update the **customers_silver** table when inserts, updates, or deletes occur, without maintaining historical records (SCD Type 1)?

  </div>
</div>



## C. Change Data Capture with AUTO CDC APIs in Apache Spark™ Declarative Pipelines

1. Run the cell below to create your starter Spark Declarative Pipeline for this demonstration. The pipeline will set the following for you:
    - Your default catalog: **labuser**
    - Your configuration parameter: `source` = `/Volumes/labuser/sdp_1_bronze/source/customers`

    **NOTES:** 
    - The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

    - If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'12 - Change Data Capture with AUTO CDC - {my_catalog}',
    root_path_folder_name="12 - Change Data Capture with AUTO CDC Project",
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['orders', 'status', 'customers'],
    configuration={'source': source_volume_path}
)

2. Complete the following steps to open the starter Spark Declarative Pipeline project for this demonstration:

   a. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

   b. In **Jobs & Pipelines** select your **12 - Change Data Capture with AUTO CDC - labuser** pipeline.

   c. In the **Pipeline details** pane on the far right, select **Open in Editor** (field to the right of **Source code**) to open the pipeline in the **Lakeflow Pipeline Editor**.

   d. In the new tab, you should see the following folders:
      - **explorations**
      - **orders**
      - **status**
      - **customers**
      - Plus the extra **python_excluded** folder that contains the Python version.

   e. Open the **customers** folder and select the **customers_pipeline.sql** file.
      - **NOTE:** The **status** and **orders** pipelines are the same as we saw in the previous demonstrations.

## D. Spark Declarative Pipeline CDC SCD Type 1 Pipeline Steps
Follow the steps below using the **customers_pipeline.sql** file in the Lakeflow Pipelines editor.

1. Run the cell below and confirm each source volume (for **orders**, **status** and **customers**) contains a single JSON file.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/orders"').display()
spark.sql(f'LIST "{source_volume_path}/status"').display()
spark.sql(f'LIST "{source_volume_path}/customers"').display()

### D1. PLEASE COMPLETE FIRST: Click the 'Run Pipeline' button to execute the Pipeline
1. To save some time, let's run the entire pipeline for **status**, **orders** and **customers**. Each volume contains 1 file.

    While the pipeline is running explore the code in the **customers/customers_pipeline.sql** for the new customers flow.

##### While the pipeline is running continue through the steps below to review the customer pipeline code.

### D2. STEP 1: JSON -> Bronze Ingestion (`customers_pipeline.sql`)
The code in **STEP 1** of the **customers_pipeline.sql** file:

   - We define a bronze streaming table named **customers_bronze_raw_demo12** using a data source configured with Auto Loader (`FROM STREAM`) to incrementally ingest files from cloud storage.

   - Adds the table property `pipelines.reset.allowed = false` to prevent deletion of all ingested bronze data if a full refresh is triggered.

   - Creates columns to capture the time of data ingestion and the source file name for each row.


### D3. STEP 2: Create the Bronze Clean Streaming Table with Data Quality Enforcement

##### **NOTE:** This displays how you can use advanced data quality techniques with expectations. Advanced expectations are outside the scope of this course.

##### The code in **STEP 2** of the **customers_pipeline.sql** file:

- Adds three violation constraint actions: **WARN**, **DROP**, and **FAIL**. Each defines how to handle constraint violations.
- Applies multiple conditions to a single constraint.
- Uses a built-in SQL function within a constraint.

##### About the data source:

- The data is a CDC feed that contains **`INSERT`**, **`UPDATE`**, and **`DELETE`** operations for customers.

  - REQUIREMENT: **UPDATE** and **INSERT** operations should contain valid entries for all fields.

  - REQUIREMENT: **DELETE** operations should contain **`NULL`** values for all fields except the **timestamp**, **customer_id**, and **operation** fields.

  - When a record is going to be dropped, all values except the **customer_id** will be `null`.

    <div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
      <table style="width: 100%; border-collapse: collapse; font-size: 14pt; line-height: 1.5;">
        <thead>
          <tr style="background: #1B5162; color: white;">
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">address</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">city</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">customer_id</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">email</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">name</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">operation</th>
            <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">state</th>
          </tr>
        </thead>
        <tbody>
          <tr style="background: #F9F7F4;">
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700; color: #0b2026;">23617</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">
              <span style="background: #FABFBA; color: #801C17; font-weight: 700; padding: 3px 10px; border-radius: 4px; font-size: 13pt;">DELETE</span>
            </td>
            <td style="padding: 10px 14px; border: 1px solid #EEEDE9; color: #618794; font-style: italic;">null</td>
          </tr>
        </tbody>
      </table>

    </div>

<br></br>
**NOTE:** To ensure only valid data reaches our silver table, we'll write a series of quality enforcement rules that allow expected null values in **DELETE** operations while rejecting bad data elsewhere.



#### We'll break down each of these constraints below:


<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.f1-cards { display: flex; gap: 8px; margin-bottom: 18px; flex-wrap: wrap; }
.f1-card {
  flex: 1; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 12px 10px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s; min-width: 140px;
}
.f1-card:hover { transform: translateY(-2px); }
.f1-card.active { background: #fff; border-left-color: var(--dc); border-right-color: var(--dc); border-bottom-color: var(--dc); }
.f1-card-label { display: block; font-size: 13pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.f1-card-sub { display: block; font-size: 11.5pt; font-weight: 400; color: #618794; margin-top: 4px; pointer-events: none; }
.f1-layout { display: flex; gap: 22px; align-items: stretch; }
.f1-code-wrap { flex: 1; position: relative; }
.f1-theme-btn {
  position: absolute; top: 10px; right: 12px; z-index: 2;
  background: rgba(255,255,255,0.12); border: 1px solid rgba(255,255,255,0.2);
  border-radius: 6px; padding: 5px 12px; font-size: 11pt; font-weight: 600; color: #cdd6f4;
  cursor: pointer; transition: background 0.15s, color 0.15s, border-color 0.15s;
}
.f1-theme-btn:hover { background: rgba(255,255,255,0.2); }
.f1-code-wrap.light .f1-theme-btn { background: rgba(0,0,0,0.06); border-color: rgba(0,0,0,0.15); color: #444; }
.f1-code-wrap.light .f1-theme-btn:hover { background: rgba(0,0,0,0.1); }
.f1-code {
  border-radius: 10px; padding: 20px 22px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12pt; line-height: 1.75; overflow-x: auto; background: #1e1e2e; color: #cdd6f4;
  transition: background 0.3s, color 0.3s;
}
.f1-code-wrap.light .f1-code { background: #fafafa; color: #383a42; }
.f1-code .tk-kw { color: #cba6f7; }
.f1-code .tk-fn { color: #89b4fa; }
.f1-code .tk-str { color: #a6e3a1; }
.f1-code .tk-num { color: #fab387; }
.f1-code .tk-cmt { color: #6c7086; }
.f1-code .tk-dim { color: #a6adc8; }
.f1-code-wrap.light .f1-code .tk-kw { color: #a626a4; }
.f1-code-wrap.light .f1-code .tk-fn { color: #4078f2; }
.f1-code-wrap.light .f1-code .tk-str { color: #50a14f; }
.f1-code-wrap.light .f1-code .tk-num { color: #986801; }
.f1-code-wrap.light .f1-code .tk-cmt { color: #a0a1a7; }
.f1-code-wrap.light .f1-code .tk-dim { color: #696c77; }
.f1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, opacity 0.25s; }
.f1-code.has-highlight .line { opacity: 0.25; }
.f1-code.has-highlight .line.hl { opacity: 1; background: rgba(255,255,255,0.08); }
.f1-code-wrap.light .f1-code.has-highlight .line.hl { background: rgba(0,0,0,0.06); }
.f1-explain { flex: 0 0 320px; display: flex; flex-direction: column; justify-content: center; }
.f1-explain-card { background: #F9F7F4; border-radius: 10px; border-top: 6px solid #ccc; padding: 20px; font-size: 14pt; line-height: 1.6; opacity: 0; transition: opacity 0.3s; min-height: 200px; }
.f1-explain-card.visible { opacity: 1; }
.f1-explain-card ul { margin: 10px 0 0 0; padding-left: 20px; }
.f1-explain-card li { margin-bottom: 10px; }
.f1-badge {
  display: inline-block; padding: 3px 10px; border-radius: 999px;
  font-size: 11.5pt; font-weight: 700; margin-bottom: 12px;
}
</style>

<!-- Header -->
<div style="background: #1B5162; color: white; border-radius: 8px 8px 4px 4px; padding: 20px 24px; margin-bottom: 18px;">
  <div style="font-size: 20pt; font-weight: 700;">Data Quality Constraints - Bronze Clean Table</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;">Each constraint defines a rule and what happens when a record violates it. Click a constraint to explore.</div>
</div>

<div class="f1-cards">
  <div class="f1-card" data-id="0" onclick="bcSelect(0)" style="--dc:#98102A; border-top-color:#98102A;">
    <span class="f1-card-label"><code>valid_id</code></span>
    <span class="f1-card-sub">FAIL UPDATE</span>
  </div><div class="f1-card" data-id="1" onclick="bcSelect(1)" style="--dc:#FF5F46; border-top-color:#FF5F46;">
    <span class="f1-card-label"><code>valid_operation</code></span>
    <span class="f1-card-sub">DROP ROW</span>
  </div><div class="f1-card" data-id="2" onclick="bcSelect(2)" style="--dc:#FFAB00; border-top-color:#FFAB00;">
    <span class="f1-card-label"><code>valid_name</code></span>
    <span class="f1-card-sub">WARN (default)</span>
  </div><div class="f1-card" data-id="3" onclick="bcSelect(3)" style="--dc:#4299E0; border-top-color:#4299E0;">
    <span class="f1-card-label"><code>valid_address</code></span>
    <span class="f1-card-sub">WARN (default)</span>
  </div><div class="f1-card" data-id="4" onclick="bcSelect(4)" style="--dc:#00A972; border-top-color:#00A972;">
    <span class="f1-card-label"><code>valid_email</code></span>
    <span class="f1-card-sub">DROP ROW</span>
  </div>
</div>

<div class="f1-layout">
  <div class="f1-code-wrap" id="bc-code-wrap">
    <button class="f1-theme-btn" id="bc-theme-btn" onclick="bcToggle()">Light Mode</button>
    <div class="f1-code" id="bc-code">
      <span class="line" data-g="all"><span class="tk-kw">CREATE STREAMING TABLE</span> 1_bronze_db.customers_bronze_clean_demo12</span>
      <span class="line" data-g="all">&nbsp;&nbsp;(</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_id</span> <span class="tk-kw">EXPECT</span> (customer_id <span class="tk-kw">IS NOT NULL</span>) <span class="tk-kw">ON VIOLATION FAIL UPDATE</span>,</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_operation</span> <span class="tk-kw">EXPECT</span> (operation <span class="tk-kw">IS NOT NULL</span>) <span class="tk-kw">ON VIOLATION DROP ROW</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_name</span> <span class="tk-kw">EXPECT</span> (name <span class="tk-kw">IS NOT NULL OR</span> operation = <span class="tk-str">"DELETE"</span>),</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_address</span> <span class="tk-kw">EXPECT</span> (</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(address <span class="tk-kw">IS NOT NULL AND</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;city <span class="tk-kw">IS NOT NULL AND</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;state <span class="tk-kw">IS NOT NULL AND</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;zip_code <span class="tk-kw">IS NOT NULL</span>) <span class="tk-kw">OR</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;operation = <span class="tk-str">"DELETE"</span>),</span>
      <span class="line" data-g="4">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">CONSTRAINT</span> <span class="tk-fn">valid_email</span> <span class="tk-kw">EXPECT</span> (</span>
      <span class="line" data-g="4">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;rlike(email, <span class="tk-str">'^([a-zA-Z0-9_\\-\\.]+)@([a-zA-Z0-9_\\-\\.]+)\\.([a-zA-Z]{2,5})$'</span>) <span class="tk-kw">OR</span></span>
      <span class="line" data-g="4">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;operation = <span class="tk-str">"DELETE"</span>) <span class="tk-kw">ON VIOLATION DROP ROW</span></span>
      <span class="line" data-g="all">&nbsp;&nbsp;)</span>
      <span class="line" data-g="all">&nbsp;&nbsp;<span class="tk-kw">COMMENT</span> <span class="tk-str">"Clean raw bronze timestamp column and add data quality constraints"</span></span>
      <span class="line" data-g="all"><span class="tk-kw">AS</span></span>
      <span class="line" data-g="all"><span class="tk-kw">SELECT</span></span>
      <span class="line" data-g="all">&nbsp;&nbsp;*,</span>
      <span class="line" data-g="all">&nbsp;&nbsp;<span class="tk-fn">CAST</span>(from_unixtime(timestamp) <span class="tk-kw">AS</span> timestamp) <span class="tk-kw">AS</span> timestamp_datetime</span>
      <span class="line" data-g="all"><span class="tk-kw">FROM STREAM</span> 1_bronze_db.customers_bronze_raw_demo12;</span>
    </div>
  </div>
  <div class="f1-explain">
    <div class="f1-explain-card" id="bc-explain-card"></div>
  </div>
</div>

</div>

<script>
var BC_DATA = [
  {
    color: '#98102A',
    badge: 'FAIL UPDATE',
    badgeBg: 'rgba(152,16,42,0.12)',
    title: 'valid_id',
    text: 'The strictest violation action. If any record arrives with a <strong>null <code>customer_id</code></strong>, the entire micro-batch update fails.',
    bullets: [
      'No records from that batch are written',
      'Use for fields that are truly non-negotiable, like a primary key',
      'Prefer this when bad data should halt the pipeline, not silently pass through'
    ]
  },
  {
    color: '#FF5F46',
    badge: 'DROP ROW',
    badgeBg: 'rgba(255,95,70,0.12)',
    title: 'valid_operation',
    text:       'Silently removes any record where <code>operation</code> is null. The pipeline keeps running and only the offending row is discarded.',
    bullets: [
      'Violation counts are tracked in pipeline metrics',
      'Use when bad rows should be excluded but not stop processing',
      'A null <code>operation</code> means we can\'t apply CDC logic — so there\'s no safe way to handle it'
    ]
  },
  {
    color: '#FFAB00',
    badge: 'WARN (default)',
    badgeBg: 'rgba(255,171,0,0.12)',
    title: 'valid_name',
    text: 'Flags records where <code>name</code> is null and the operation is not a DELETE. No rows are dropped — violations are tracked in metrics only.',
    bullets: [
      'No <code>ON VIOLATION</code> clause = WARN behavior by default',
      'The <code>OR operation = "DELETE"</code> allows expected nulls for deletes',
      'Use when you want visibility into data quality issues without disrupting the pipeline'
    ]
  },
  {
    color: '#4299E0',
    badge: 'WARN (default)',
    badgeBg: 'rgba(66,153,224,0.12)',
    title: 'valid_address',
    text: 'Checks all four address fields at once. A record passes if all four are non-null, OR if the operation is DELETE (where nulls are expected).',
    bullets: [
      'Multiple conditions combined with AND in a single constraint',
      'The OR short-circuit lets DELETE records bypass address validation',
      'Violations are logged to metrics but rows are not dropped'
    ]
  },
  {
    color: '#00A972',
    badge: 'DROP ROW',
    badgeBg: 'rgba(0,169,114,0.12)',
    title: 'valid_email',
    text: 'Uses <code>rlike()</code>, a built-in SQL regex function to validate email format. Records with an invalid email are dropped.',
    bullets: [
      'The regex pattern matches standard email formats',
      'DELETE operations are exempt (their email field will be null)',
      'Dropped records will have all fields null except <code>customer_id</code>'
    ]
  }
];

var bcCurrent = null, bcLight = false;

function bcToggle() {
  bcLight = !bcLight;
  document.getElementById('bc-code-wrap').classList.toggle('light', bcLight);
  document.getElementById('bc-theme-btn').textContent = bcLight ? 'Dark Mode' : 'Light Mode';
}

function bcSelect(id) {
  var code = document.getElementById('bc-code');
  var card = document.getElementById('bc-explain-card');
  document.querySelectorAll('.f1-card').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id);
  });
  if (bcCurrent === id) {
    code.classList.remove('has-highlight');
    card.classList.remove('visible');
    document.querySelectorAll('.f1-card').forEach(function(b) { b.classList.remove('active'); });
    bcCurrent = null;
    return;
  }
  bcCurrent = id;
  var c = BC_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln) {
    var g = ln.dataset.g;
    ln.classList.toggle('hl', g === String(id) || g === 'all');
  });
  var bulletsHtml = c.bullets.map(function(b) { return '<li>' + b + '</li>'; }).join('');
  card.style.borderTopColor = c.color;
  card.innerHTML =
    '<span class="f1-badge" style="background:' + c.badgeBg + ';color:' + c.color + ';">' + c.badge + '</span>' +
    '<div style="font-size:17pt;font-weight:700;margin-bottom:10px;color:#0b2026;font-family:monospace;">' + c.title + '</div>' +
    '<div style="margin-bottom:10px;">' + c.text + '</div>' +
    '<ul>' + bulletsHtml + '</ul>';
  card.classList.add('visible');
}
</script>

### D4. STEP 3: Processing CDC Data with **`AUTO CDC INTO`**
Spark Declarative Pipelines introduces a new syntactic structure for simplifying CDC feed processing: `AUTO CDC INTO` (formerly `APPLY CHANGES INTO`).

The code in **STEP 3** of the **customers_pipeline.sql** file uses `AUTO CDC INTO` to:

- **CREATES** 
  - **sdp_2_silver.scd_type_1_customers_silver_demo12** streaming table if it doesn't exist,

- **UPDATES** 
  - **sdp_2_silver.scd_type_1_customers_silver_demo12** streaming table with updates, inserts and deletes using records from the **sdp_1_bronze.customers_bronze_clean_demo12** streaming table.

#### Additional Notes
**`AUTO CDC INTO`** has the following guarantees and requirements:
- Performs incremental/streaming ingestion of CDC data
- Provides simple syntax to specify one or many fields as the primary key for a table
- Default assumption is that rows will contain inserts and updates
- Can optionally apply deletes
- Automatically orders late-arriving records using user-provided sequencing key (order to process rows)
- Uses a simple syntax for specifying columns to ignore with the **`EXCEPT`** keyword
- The default to applying changes is SCD Type 1. You can also use SCD Type 2 if you would like. We will focus on SCD Type 1.


#### Documentation
AUTO CDC INTO (Apache Spark™ Declarative Pipelines):
[AWS](https://docs.databricks.com/aws/en/dlt-ref/dlt-sql-ref-apply-changes-into) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-apply-changes-into) |
[GCP](https://docs.databricks.com/gcp/en/dlt-ref/dlt-sql-ref-apply-changes-into)

The AUTO CDC APIs - Simplify change data capture with Apache Spark™ Declarative Pipelines:
[AWS](https://docs.databricks.com/aws/en/dlt/cdc) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc) |
[GCP](https://docs.databricks.com/gcp/en/dlt/cdc)

### D5. STEP 4: Explore the Customers Pipeline Graph
After running the pipeline and reviewing the code cells, take time to explore the pipeline results for the **customers** flow following the steps below.

**Run with 1 JSON File**

![demo12_cdc_run01.png](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/change-data-capture/demo12_cdc_run_1.png)



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    View the Results
  </strong>
  <div style="color:#333;">

Notice the following:
1. In the **customers** flow in the pipeline graph, notice that **939** rows were streamed into the three streaming tables.
    - This is because all records are new and valid entries, they were ingested throughout the flow.

2. In the table window below, find the **scd_type_1_customers_silver_demo12** table and select **Table metrics**. 

    Note the following:

    - The **Upserted** column indicates that all **939** rows were upserted into the table, as all rows are new.
  </div>
</div>


### D6. STEP 5: Explore the Customers Pipeline Tables

1. Run the query below to view the **scd_type_1_customers_silver_demo12** streaming table (the table with SCD Type 1 updates, inserts and deletes).

    Notice the following after the first run ingestion the **00.json** file:

   - The streaming table contains all **939 rows** from the **00.json** file, since they are all new customers being added to the target table.

   - Each record was inserted into the empty streaming table.

In [0]:
SELECT *
FROM sdp_2_silver.scd_type_1_customers_silver_demo12;

2. Query the **scd_type_1_customers_silver_demo12** streaming table for the following **customer_id** values (*23225*, *23617*).

   Notice the following:
      - **customer_id** = *23225*
         - **Address**: `76814 Jacqueline Mountains Suite 815`
         - **State**: `TX`
      - **customer_id** = *23617*
         - This customer exists in the first execution (in file **00.json**)

In [0]:
SELECT *
FROM sdp_2_silver.scd_type_1_customers_silver_demo12
WHERE customer_id IN (23225, 23617);

## E. Land New Data to Your Data Source Volume
Complete the following after executing and reviewing the **customers** pipeline flow that consisted of ingesting one file (**00.json**) from cloud storage.

1. Run the cell below to land a new JSON file to each volume (**customers**, **status** and **orders**) to simulate new files being added to your cloud storage locations.

In [0]:
%python

## Find data in workspace data folder
data_path = "/Volumes/dbacademy/default/data"

## Land JSON files to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/orders',
    target_volume_path=f'{source_volume_path}/orders',
    n=2
)

## Land JSON files to your status volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/status',
    target_volume_path=f'{source_volume_path}/status',
    n=2
)

## Land JSON files to your customers volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/customers',
    target_volume_path=f'{source_volume_path}/customers',
    n=2
)

2. Run the cell below to programmatically view the files in your `/Volumes/labuser/sdp_1_bronze/source/customers` volume.

    Confirm your volume now contains the original **00.json** file and the new **01.json** file.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/customers"').display()

3. Run the cell to explore the raw data in the new **01.json** file prior to ingesting it in your pipeline.

   Notice the following:

   - This file contains **23** rows.

   - The **operation** column specifies **UPDATE**, **DELETE**, and **NEW** operations for customers.
      - **In the new 01.json file there are**:
         - 12 customers with **UPDATE** values
         - 1 customer with a **DELETE** value
         - 10 new customers with a **NEW** value

In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/customers/01.json',
  format => "JSON"
)
ORDER BY customer_id;

4. Run the cell to view **customer_id** values *23225* and *23617* in the **01.json** file.

   - In the results below, find the row with **customer_id** *23225* and note the following:

      - The original address for **Sandy Adams** (from the streaming table, file **00.json**) was: `76814 Jacqueline Mountains Suite 815`, `TX`
      - The updated address for **Sandy Adams** (from the file below) is: `512 John Stravenue Suite 239`, `TN`

   - In the results below, find the row with **customer_id** *23617* and note the following:
      - The **operation** for this customer is **DELETE**.
      - When the **operation** column is delete, all other column values are `null`.


In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/customers/01.json',
  format => "JSON"
)
WHERE customer_id IN (23225, 23617)
ORDER BY customer_id;

### E1. Run the SDP with the New File

##### Go back to your pipeline and click `Run pipeline` button to ingest the new JSON file (**01.json**) incrementally and perform CDC SCD Type 1 on the `scd_type_1_customers_silver_demo12` table.

## F. Explore the Customers Pipeline

After you have explored and landed 1 new JSON file into each of your cloud data sources, complete the following to explore the **customers** flow in the **Pipeline graph**:

a. 23 rows were read into the:

  - **customers_bronze_raw_demo12** streaming table
  - **customers_bronze_clean_demo12** streaming table (all data quality checks passed)
  - The pipeline only ingested and processed the NEW **01.json** file

b. In the **scd_type_1_customers_silver_demo12** streaming table details (The CDC SCD Type 1 table) it contains:
  - **Upserted = 22**:
    - 12 customers with UPDATE values (previous customer were simply updated with the new values)
    - 10 new customers with a NEW value (new customers were inserted into the table)
  - **Deleted records = 1**:
    - 1 customer was marked as DELETE and deleted from the table

![Run 2](https://files.training.databricks.com/binder/prod_main/build-data-pipelines-with-apache-spark-declarative-pipelines-en_us-3.2.1/images/20260902T093127Z/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/images/change-data-capture/demo12_cdc_run_2.png)

## G. Explore the CDC SCD Type 1 on the `scd_type_1_customers_silver_demo12` Streaming Table

1. View the data in the **scd_type_1_customers_silver_demo12** streaming table with SCD Type 1 and observe the following:

   a. The table contains **948 rows**:
      - **initial 939 customers**
      - \+ **10** new customers
      - \- **1** deleted customer
      - **NOTES:**
         - The **12** updates to original customers were made in place and updated the original record (SCD Type 1 does not keep historical records).
         - The **1** record marked for deletion was deleted from the table.

In [0]:
SELECT customer_id, address, name
FROM sdp_2_silver.scd_type_1_customers_silver_demo12;

2. Query the **sdp_2_silver.scd_type_1_customers_silver_demo12** table for the following **customer_id** values: *23225* and *23617*. These were the values we reviewed earlier.

    Notice the following:

    - **customer_id** *23225* has been updated to the new address. The historical address was not retained because we used SCD Type 1.
    - **customer_id** *23617* has been deleted from the table. It no longer exists because we used SCD Type 1.


In [0]:
SELECT *
FROM sdp_2_silver.scd_type_1_customers_silver_demo12
WHERE customer_id IN (23225, 23617);

## Additional Resources

- What is change data capture (CDC)?:
[AWS](https://docs.databricks.com/aws/en/dlt/what-is-change-data-capture) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/what-is-change-data-capture) |
[GCP](https://docs.databricks.com/gcp/en/dlt/what-is-change-data-capture)

- AUTO CDC INTO (Apache Spark™ Declarative Pipelines) documentation:
[AWS](https://docs.databricks.com/aws/en/dlt-ref/dlt-sql-ref-apply-changes-into) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-apply-changes-into) |
[GCP](https://docs.databricks.com/gcp/en/dlt-ref/dlt-sql-ref-apply-changes-into)

- The AUTO CDC APIs - Simplify change data capture with Apache Spark™ Declarative Pipelines documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/cdc) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc) |
[GCP](https://docs.databricks.com/gcp/en/dlt/cdc)

- [How to implement Slowly Changing Dimensions when you have duplicates - Part 1: What to look out for?](https://community.databricks.com/t5/technical-blog/how-to-implement-slowly-changing-dimensions-when-you-have/ba-p/40568)


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
